# Swin3D-DNP Test Runner

This notebook runs the test suite on Colab with GPU acceleration.

**Features:**
- GPU-accelerated testing
- Optional push of test results back to git
- Coverage reporting

## Setup

1. Open in Colab: `File > Open notebook > GitHub`
2. Enable GPU: `Runtime > Change runtime type > GPU`
3. Run all cells

In [ ]:
# @title Configuration
REPO_URL = "https://github.com/maxrusse/Swin3d-DNP.git"  # @param {type:"string"}
BRANCH = "claude/implement-swin3d-dnp-GFsJW"  # @param {type:"string"}
PUSH_RESULTS = False  # @param {type:"boolean"}
GITHUB_TOKEN = ""  # @param {type:"string"}

In [ ]:
# @title Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

In [ ]:
# @title Clone repository
import os

# Clone repo
if os.path.exists("Swin3d-DNP"):
    !rm -rf Swin3d-DNP

!git clone {REPO_URL}
%cd Swin3d-DNP
!git checkout {BRANCH}
!git pull origin {BRANCH}

In [ ]:
# @title Install dependencies
!pip install -e . -q
!pip install pytest pytest-cov -q
print("Dependencies installed!")

In [ ]:
# @title Run all tests
import datetime

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
result_file = f"test_results_{timestamp}.txt"

# Run tests with coverage
!python -m pytest tests/ -v --tb=short --cov=swin3d_dnp --cov-report=term 2>&1 | tee {result_file}

print(f"\nResults saved to: {result_file}")

In [ ]:
# @title Run specific test modules (optional)
# Uncomment the tests you want to run

# Geometry tests
# !python -m pytest tests/test_geometry.py -v

# Model tests (requires more GPU memory)
# !python -m pytest tests/test_models.py -v

# Loss function tests
# !python -m pytest tests/test_losses.py -v

# Inference tests (NMS, etc.)
# !python -m pytest tests/test_inference.py -v

In [ ]:
# @title Push results to Git (optional)
if PUSH_RESULTS and GITHUB_TOKEN:
    # Configure git
    !git config user.email "colab-runner@example.com"
    !git config user.name "Colab Test Runner"

    # Set remote with token
    repo_with_token = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
    !git remote set-url origin {repo_with_token}

    # Create results directory
    !mkdir -p test_results
    !mv test_results_*.txt test_results/

    # Commit and push
    !git add test_results/
    !git commit -m "Add test results from Colab run"
    !git push origin {BRANCH}

    print("Results pushed to repository!")
else:
    print("Skipping push (PUSH_RESULTS=False or no token provided)")

In [ ]:
# @title GPU Memory Test (optional)
import torch
import gc

def test_memory(batch_size, spatial_size):
    """Test GPU memory with synthetic forward pass."""
    try:
        gc.collect()
        torch.cuda.empty_cache()

        # Simulate coarse input
        x = torch.randn(batch_size, 1, spatial_size, spatial_size, spatial_size, device='cuda')

        # Memory allocated
        mem_mb = torch.cuda.memory_allocated() / 1024**2
        print(f"Batch={batch_size}, Size={spatial_size}³: {mem_mb:.1f} MB allocated")

        del x
        torch.cuda.empty_cache()
        return True
    except RuntimeError as e:
        print(f"Batch={batch_size}, Size={spatial_size}³: OOM")
        return False

if torch.cuda.is_available():
    print("Testing GPU memory limits...")
    for bs in [1, 2, 4]:
        for size in [64, 96, 128]:
            test_memory(bs, size)
else:
    print("No GPU available - skipping memory test")

## Summary

After running:
1. Check test output above for any failures
2. Review coverage report
3. If `PUSH_RESULTS=True`, results are saved to the repo

### Troubleshooting

- **OOM errors**: Reduce batch size or spatial dimensions
- **Import errors**: Re-run the install cell
- **Git push fails**: Check GITHUB_TOKEN has write access